In [2]:
pip install netCDF4 numpy xarray rioxarray pyresample


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import glob
from collections import defaultdict
import numpy as np
import xarray as xr
import netCDF4 as nc
import rioxarray
from pyresample import geometry, kd_tree

# ==============================================================================
# 1. FOLDER & REGION SETTINGS
# ==============================================================================
L2_DIR = r"E:\Talif_Mia\Converted"         # Folder containing your downloaded .L2.OC.nc files
QGIS_OUT_DIR =r"E:\Talif_Mia\Converted\QGIS_1km"  # Output folder for QGIS-ready 1 km GeoTIFFs
os.makedirs(QGIS_OUT_DIR, exist_ok=True)

# Gulf of Alaska bounding box: (West, South, East, North)
BBOX = (-165.0, 52.0, -130.0, 62.0)

# All 10 MODIS Rrs bands required by your methodology
RRS_BANDS = [
    "Rrs_412", "Rrs_443", "Rrs_469", "Rrs_488", "Rrs_531",
    "Rrs_547", "Rrs_555", "Rrs_645", "Rrs_667", "Rrs_678"
]

# Standard NASA L2 quality flag bitmask (excludes clouds, land, sun glint, straylight, etc.)
EXCLUDE_BITS = [0, 1, 3, 4, 8, 9, 10, 12, 14, 15, 16, 19, 21, 22, 25, 26]
FLAG_MASK = sum(1 << b for b in EXCLUDE_BITS)

# ==============================================================================
# 2. DEFINE 1 KM TARGET GRID OVER THE GULF OF ALASKA
# ==============================================================================
grid_lats = np.arange(BBOX[3], BBOX[1], -0.009)   # ~1 km North-to-South
grid_lons = np.arange(BBOX[0], BBOX[2], 0.0165)   # ~1 km West-to-East at 57°N
lon_2d, lat_2d = np.meshgrid(grid_lons, grid_lats)
target_area = geometry.SwathDefinition(lons=lon_2d, lats=lat_2d)

# ==============================================================================
# 3. GROUP DOWNLOADED SWATHS BY DATE & CONVERT TO 1 KM GEOTIFFS
# ==============================================================================
files_by_date = defaultdict(list)
for filepath in sorted(glob.glob(os.path.join(L2_DIR, "*.nc"))):
    # Extracts YYYYMMDD from filenames like AQUA_MODIS.20230515T215501.L2.OC.nc
    date_str = os.path.basename(filepath).split(".")[1][:8]
    files_by_date[date_str].append(filepath)

print(f"Found {len(files_by_date)} unique date(s) to convert.")

for date_str, day_files in files_by_date.items():
    day_folder = os.path.join(QGIS_OUT_DIR, date_str)
    os.makedirs(day_folder, exist_ok=True)

    sums = {b: np.zeros(lat_2d.shape, dtype=np.float64) for b in RRS_BANDS}
    counts = {b: np.zeros(lat_2d.shape, dtype=np.int32) for b in RRS_BANDS}

    for f in day_files:
        with nc.Dataset(f, "r") as ds:
            lats = ds.groups["navigation_data"].variables["latitude"][:].filled(np.nan)
            lons = ds.groups["navigation_data"].variables["longitude"][:].filled(np.nan)
            flags = ds.groups["geophysical_data"].variables["l2_flags"][:].filled(0).astype(np.uint32)
            bad_pixels = (flags & FLAG_MASK) != 0

            band_stack = []
            for b in RRS_BANDS:
                arr = ds.groups["geophysical_data"].variables[b][:].astype(np.float32)
                if np.ma.isMaskedArray(arr):
                    arr = arr.filled(np.nan)
                arr[bad_pixels | (arr <= 0)] = np.nan
                band_stack.append(arr)

        swath = geometry.SwathDefinition(lons=lons, lats=lats)
        resampled = kd_tree.resample_nearest(
            swath,
            np.dstack(band_stack),
            target_area,
            radius_of_influence=1500,
            fill_value=np.nan
        )

        for idx, b in enumerate(RRS_BANDS):
            layer = resampled[:, :, idx]
            valid = ~np.isnan(layer)
            sums[b][valid] += layer[valid]
            counts[b][valid] += 1

    # Save each of the 10 bands as a standalone 1 km GeoTIFF for QGIS
    for b in RRS_BANDS:
        daily_grid = np.full(lat_2d.shape, np.nan, dtype=np.float32)
        valid = counts[b] > 0
        daily_grid[valid] = (sums[b][valid] / counts[b][valid]).astype(np.float32)

        da = xr.DataArray(
            daily_grid,
            coords={"y": grid_lats, "x": grid_lons},
            dims=("y", "x"),
            name=b
        )
        da = da.rio.write_crs("EPSG:4326")
        tif_path = os.path.join(day_folder, f"{b}_{date_str}_1km.tif")
        da.rio.to_raster(tif_path, compress="LZW")

    print(f"[{date_str}] Exported 10 GeoTIFFs -> {day_folder}")

Found 1 unique date(s) to convert.
[nc] Exported 10 GeoTIFFs -> E:\Talif_Mia\Converted\QGIS_1km\nc
